# 09 — Spectral transformer on the mock cube
## Part A · From a spectrum to a stress logit

Build and check a small PyTorch encoder before the second day's training. Each token represents one measured wavelength of a centre pixel. Notebook 04's fixed band selection preserves physical wavelengths and their gaps. Notebook 06 supplies the spatial split convention. Run notebooks 03 and 04 first.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch import nn

root = Path(".") if Path("notebooks").is_dir() else Path("..")
with np.load(root / "data/processed/mock_pca.npz") as archive:
    valid = archive["valid"]
    kept_nm = archive["wavelengths_kept"]
with h5py.File(root / "data/mock/mock_cube.h5") as scene:
    wavelengths = scene["wavelengths_nm"][...]
    keep = np.isin(wavelengths, kept_nm)
    assert np.array_equal(wavelengths[keep], kept_nm)
    spectra = scene["cube"][...][..., keep]
    labels = scene["labels"][...]
assert spectra.shape[:2] == valid.shape == labels.shape
split_row = 38
train_spectra = spectra[:split_row][valid[:split_row]]
train_labels = labels[:split_row][valid[:split_row]]
mean = train_spectra.mean(axis=0, dtype=np.float64)
scale = train_spectra.std(axis=0, dtype=np.float64).clip(min=1e-8)
train_x = torch.from_numpy(((train_spectra - mean) / scale).astype(np.float32))
train_y = torch.from_numpy(train_labels.astype(np.float32))
print(f"{len(train_x)} training centres, {len(kept_nm)} wavelengths, {int(train_y.sum())} stressed centres")

2546 training centres, 166 wavelengths, 8 stressed centres


### Embed each wavelength

A token carries two values: standardised reflectance and its wavelength rescaled to the interval [−1, 1]. A linear projection embeds the pair in 32 dimensions. Wavelength identifies a token even across the water-absorption gaps removed in notebook 04. Four attention heads let tokens exchange information across the spectrum. Mean pooling then gives one vector per centre pixel.

In [2]:
class SpectralTransformer(nn.Module):
    def __init__(self, wavelengths_nm, width=32, heads=4):
        super().__init__()
        wave = torch.as_tensor(wavelengths_nm, dtype=torch.float32)
        assert wave.ndim == 1 and torch.all(torch.diff(wave) > 0)
        positions = 2 * (wave - wave.min()) / (wave.max() - wave.min()) - 1
        self.register_buffer("positions", positions)
        self.embedding = nn.Linear(2, width)
        block = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=2 * width,
            dropout=0.1, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=1, enable_nested_tensor=False)
        self.classifier = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))

    def tokens(self, spectrum):
        if spectrum.ndim != 2 or spectrum.shape[1] != len(self.positions):
            raise ValueError("Expected one reflectance value per retained wavelength.")
        pairs = torch.stack((spectrum, self.positions.expand_as(spectrum)), dim=-1)
        return self.embedding(pairs)

    def forward(self, spectrum):
        features = self.encoder(self.tokens(spectrum))
        return self.classifier(features.mean(dim=1)).squeeze(-1)


torch.manual_seed(0)
model = SpectralTransformer(kept_nm)
indices = torch.cat([torch.where(train_y == k)[0][:4] for k in (0, 1)])
xb, yb = train_x[indices], train_y[indices]
assert torch.isfinite(xb).all() and xb.shape == (8, len(kept_nm))
print("Spectra", tuple(xb.shape), "| tokens", tuple(model.tokens(xb).shape), "| logits", tuple(model(xb).shape))
print("Trainable parameters", sum(p.numel() for p in model.parameters() if p.requires_grad))

Spectra (8, 166) | tokens (8, 166, 32) | logits (8,)
Trainable parameters 8737


### Check attention and gradients

In evaluation mode, inspect the attention weights for one spectrum. Each head assigns a distribution over input wavelengths for every query wavelength. Then run a single backward pass on eight training examples to verify that the loss reaches the embedding, encoder and classifier. This is a wiring check.

In [3]:
model.eval()
block = model.encoder.layers[0]
with torch.no_grad():
    token = model.tokens(xb[:1])
    _, weights = block.self_attn(token, token, token, need_weights=True, average_attn_weights=False)
    assert weights.shape == (1, 4, len(kept_nm), len(kept_nm))
    torch.testing.assert_close(weights.sum(-1), torch.ones_like(weights.sum(-1)))
    torch.testing.assert_close(model(xb)[:1], model(xb[:1]), atol=1e-5, rtol=1e-5)
print("Attention (batch, heads, query bands, key bands)", tuple(weights.shape))

model.train()
model.zero_grad(set_to_none=True)
logits = model(xb)
assert logits.shape == yb.shape
loss = nn.BCEWithLogitsLoss()(logits, yb)
loss.backward()
for name, param in model.named_parameters():
    assert param.grad is not None and torch.isfinite(param.grad).all() and param.grad.norm() > 0, name
print(f"Finite loss {loss.item():.4f}; every trainable parameter received a gradient.")

Attention (batch, heads, query bands, key bands) (1, 4, 166, 166)
Finite loss 0.7964; every trainable parameter received a gradient.


## Part B - Train and evaluate the spectral transformer

Use valid pixels from rows 0-37 for training and rows 38 onward for validation, matching notebooks 06 and 08. Apply the training means and scales from Part A to each validation spectrum. Fix the epoch count and decision threshold before scoring the held-out region.

In [4]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_fscore_support

val_mask = valid[split_row:]
val_raw = spectra[split_row:][val_mask]
val_x = torch.from_numpy(((val_raw - mean) / scale).astype(np.float32))
val_y = labels[split_row:][val_mask]
assert len(train_y) + len(val_y) == valid.sum()
assert torch.isfinite(val_x).all()
for name, targets in (("Training", train_labels), ("Validation", val_y)):
    healthy, stressed = np.bincount(targets, minlength=2)
    assert healthy > 0 and stressed > 0
    print(f"{name}: {healthy} healthy, {stressed} stressed")

Training: 2538 healthy, 8 stressed
Validation: 1930 healthy, 13 stressed


### Fit a fresh encoder

Use four complete passes through the training set with Adam and batches of 32 spectra. Weight each stressed example by the healthy-to-stressed count ratio calculated on training pixels. The small batch bounds memory for the 166-token attention calculation.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
net = SpectralTransformer(kept_nm).to(device)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True,
                    generator=torch.Generator().manual_seed(0))
positive_weight = (train_y == 0).sum() / train_y.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight.to(device))
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
print(f"Device {device}, positive weight {positive_weight.item():.2f}")

for epoch in range(1, 5):
    net.train()
    total = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(net(xb.to(device)), yb.to(device))
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
    assert np.isfinite(total)
    print(f"Epoch {epoch}: weighted training loss {total / len(train_y):.4f}")

Device cpu, positive weight 317.25


Epoch 1: weighted training loss 0.5563


Epoch 2: weighted training loss 0.0492


Epoch 3: weighted training loss 0.0264


Epoch 4: weighted training loss 0.0184


### Evaluate held-out centres

Predict stress for logits greater than zero. Precision, recall and F1 measure decisions at this threshold. Average precision (AP) measures the ranking across thresholds.

In [6]:
net.eval()
with torch.inference_mode():
    scores = torch.cat([net(batch.to(device)).cpu() for batch in val_x.split(32)]).numpy()
assert scores.shape == val_y.shape and np.isfinite(scores).all()
pred = scores > 0
tn, fp, fn, tp = confusion_matrix(val_y, pred, labels=[0, 1]).ravel()
precision, recall, f1, _ = precision_recall_fscore_support(
    val_y, pred, average="binary", zero_division=0
)
ap = average_precision_score(val_y, scores)
print(f"TP {tp} | FP {fp} | FN {fn} | TN {tn}")
print(f"Precision {precision:.3f} | recall {recall:.3f} | F1 {f1:.3f} | AP {ap:.3f}")
print(f"Validation prevalence {val_y.mean():.3%}")

TP 13 | FP 0 | FN 0 | TN 1930
Precision 1.000 | recall 1.000 | F1 1.000 | AP 1.000
Validation prevalence 0.669%


### Save and verify the inference checkpoint

Store the wavelengths, training statistics, encoder configuration and decision threshold with the weights. Reload on CPU and reproduce the first validation batch starting from reflectance.

In [7]:
path = root / "models/spectral_transformer_mock.pt"
path.parent.mkdir(exist_ok=True)
torch.save({
    "state_dict": net.state_dict(), "width": 32, "heads": 4,
    "wavelengths_nm": torch.from_numpy(kept_nm),
    "band_mean": torch.from_numpy(mean), "band_scale": torch.from_numpy(scale),
    "threshold_logit": 0.0, "split_row": split_row,
}, path)
saved = torch.load(path, map_location="cpu", weights_only=True)
restored = SpectralTransformer(saved["wavelengths_nm"], saved["width"], saved["heads"])
restored.load_state_dict(saved["state_dict"])
restored.eval()
raw = torch.from_numpy(val_raw[:32]).double()
inputs = ((raw - saved["band_mean"]) / saved["band_scale"]).float()
with torch.inference_mode():
    torch.testing.assert_close(restored(inputs), torch.from_numpy(scores[:len(inputs)]),
                               rtol=1e-4, atol=1e-4)
print(f"Verified {path.name} on {len(inputs)} held-out spectra.")

Verified spectral_transformer_mock.pt on 32 held-out spectra.


### Comparison with notebook 08

At the fixed threshold, the transformer found all 13 stressed validation pixels and flagged zero of the 1,930 healthy pixels. The PCA + Random Forest baseline made the same decisions. HybridSN also found all 13 stressed pixels and raised 59 false alarms. All three achieved average precision of 1.000 on this split.

The training set contains eight stressed pixels from the same simulated spot as the validation positives. These results demonstrate within-scene classification. Performance on an independent field or stress event remains unmeasured.